### Importing Libraries

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import json
import sqlite3
import requests
from ydata_profiling import ProfileReport
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer

## Part B: Data Acquisition 

### 3. Import Datasets from multiple sources

In [61]:
# load csv files
main_transaction = pd.read_csv('data/main_transactions.csv')
main_transaction.head()

,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,default_flag
0,CUST1000,821240.738204,Home,39,13.20,0
1,CUST1001,166721.999353,Education,18,29.43,0
2,CUST1002,371720.170500,Business,114,14.89,0
3,CUST1003,367386.291190,Car,8,9.65,0
4,CUST1004,97156.360370,Education,149,29.25,0


In [62]:
# parse the JSON file
with open('data/customer_metadata.json', 'r') as f:
    customer_metadata = json.load(f)

print(customer_metadata)

[{'customer_id': 'CUST1000', 'age': 56.0, 'gender': 'Male', 'region': 'West', 'education_level': 'Secondary', 'employment_type': 'Salaried', 'join_date': '2021-05-24'}, {'customer_id': 'CUST1001', 'age': 69.0, 'gender': 'Male', 'region': 'North', 'education_level': 'Graduate', 'employment_type': 'Self-Employed', 'join_date': '2021-10-23'}, {'customer_id': 'CUST1002', 'age': 46.0, 'gender': 'Other', 'region': 'North', 'education_level': 'Graduate', 'employment_type': 'Salaried', 'join_date': '2018-04-27'}, {'customer_id': 'CUST1003', 'age': 32.0, 'gender': 'Female', 'region': 'North', 'education_level': 'Graduate', 'employment_type': 'Salaried', 'join_date': '2021-04-17'}, {'customer_id': 'CUST1004', 'age': nan, 'gender': 'Male', 'region': 'North', 'education_level': 'Post-Graduate', 'employment_type': 'Salaried', 'join_date': '2021-06-23'}, {'customer_id': 'CUST1005', 'age': 25.0, 'gender': 'Female', 'region': 'West', 'education_level': 'Secondary', 'employment_type': 'Self-Employed', 

In [63]:
# fetch recorrds from sql
conn = sqlite3.connect("data/repayment_history.db")
repayment_data = pd.read_sql("SELECT * FROM repayment_data", conn)
print(repayment_data.head())

  customer_id  annual_income  credit_score  repayment_history
0    CUST1000  841473.298565    663.935111                  3
1    CUST1001  179168.069828    714.909764                  2
2    CUST1002            NaN    635.226534                  1
3    CUST1003  311056.808239    582.369296                  0
4    CUST1004            NaN    554.746159                  3


In [64]:
# fetch data from the API (external economic indicators)
response = requests.get("https://bytebin.lucko.me/0W2HQJlJKr")
economic_data = response.json()
print(economic_data)

[{'customer_id': 'CUST1000', 'region': 'West', 'external_inflation_rate': 4.5, 'external_unemployment_rate': 4.8, 'economic_risk_score': 0.32}, {'customer_id': 'CUST1001', 'region': 'North', 'external_inflation_rate': 5.4, 'external_unemployment_rate': 6.2, 'economic_risk_score': 0.45}, {'customer_id': 'CUST1002', 'region': 'North', 'external_inflation_rate': 5.4, 'external_unemployment_rate': 6.2, 'economic_risk_score': 0.45}, {'customer_id': 'CUST1003', 'region': 'North', 'external_inflation_rate': 5.4, 'external_unemployment_rate': 6.2, 'economic_risk_score': 0.45}, {'customer_id': 'CUST1004', 'region': 'North', 'external_inflation_rate': 5.4, 'external_unemployment_rate': 6.2, 'economic_risk_score': 0.45}, {'customer_id': 'CUST1005', 'region': 'West', 'external_inflation_rate': 4.5, 'external_unemployment_rate': 4.8, 'economic_risk_score': 0.32}, {'customer_id': 'CUST1006', 'region': 'South', 'external_inflation_rate': 4.8, 'external_unemployment_rate': 5.1, 'economic_risk_score': 

In [65]:
# load main dataset
df=pd.read_csv('data/credit_risk_dataset.csv')
df.head()

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,CUST1000,56.0,Male,West,Secondary,Salaried,841473.298565,821240.738204,Home,663.935111,3,39,13.20,2021-05-24,0
1,CUST1001,69.0,Male,North,Graduate,Self-Employed,179168.069828,166721.999353,Education,714.909764,2,18,29.43,2021-10-23,0
2,CUST1002,46.0,Other,North,Graduate,Salaried,NaN,371720.170500,Business,635.226534,1,114,14.89,2018-04-27,0
3,CUST1003,32.0,Female,North,Graduate,Salaried,311056.808239,367386.291190,Car,582.369296,0,8,9.65,2021-04-17,0
4,CUST1004,NaN,Male,North,Post-Graduate,Salaried,NaN,97156.360370,Education,554.746159,3,149,29.25,2021-06-23,0


## Part C: Data Understanding & Cleaning

### 4. Explore the dataset using pandas

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1000 non-null   object 
 1   age                922 non-null    float64
 2   gender             953 non-null    object 
 3   region             1000 non-null   object 
 4   education_level    1000 non-null   object 
 5   employment_type    937 non-null    object 
 6   annual_income      917 non-null    float64
 7   loan_amount        1000 non-null   float64
 8   loan_purpose       1000 non-null   object 
 9   credit_score       888 non-null    float64
 10  repayment_history  1000 non-null   int64  
 11  transaction_count  1000 non-null   int64  
 12  spending_ratio     1000 non-null   float64
 13  join_date          1000 non-null   object 
 14  default_flag       1000 non-null   int64  
dtypes: float64(5), int64(3), object(7)
memory usage: 117.3+ KB


In [67]:
df.describe()

,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,default_flag
count,922.000000,9.170000e+02,1.000000e+03,888.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,43.786334,1.790538e+06,4.350655e+05,675.989490,1.267000,91.892000,18.042440,0.037000
std,14.982614,1.095508e+07,1.724177e+06,84.108116,1.118464,50.098925,9.263572,0.188856
min,18.000000,8.689297e+04,2.061440e+04,150.000000,0.000000,5.000000,4.190000,0.000000
25%,31.000000,2.877050e+05,9.754397e+04,625.212264,0.000000,48.000000,11.275000,0.000000
50%,44.000000,4.363368e+05,1.990802e+05,678.550946,1.000000,91.000000,16.220000,0.000000
75%,56.000000,7.011574e+05,3.838792e+05,727.871943,2.000000,136.000000,22.352500,0.000000
max,69.000000,1.372205e+08,2.401228e+07,990.000000,6.000000,179.000000,82.110000,1.000000


### 5. Perform Pandas Profiling to generate a data quality report

In [68]:

# profile = ProfileReport(df, title="Pandas Profiling Report", explorative=True)
# profile.to_file("report/data_quality_report.html")

### 6. Handling Missing data with:

#### 6.1 Simple Imputer (numerical)

In [69]:
print(f'before imputation, missing values in age column: {df["age"].isnull().sum()}')
age_imputer = SimpleImputer(strategy='mean')
df['age'] = age_imputer.fit_transform(df[['age']])
print(f'after imputation, missing values in age column: {df["age"].isnull().sum()}')

before imputation, missing values in age column: 78
after imputation, missing values in age column: 0


#### 6.2 Simple Imputer (categorical)

In [70]:
print(f'before imputation, missing values in employment_type column: {df["employment_type"].isnull().sum()}')
employment_imputer = SimpleImputer(strategy='most_frequent')
df['employment_type'] = employment_imputer.fit_transform(df[['employment_type']]).ravel()
print(f'after imputation, missing values in employment_type column: {df["employment_type"].isnull().sum()}')

before imputation, missing values in employment_type column: 63
after imputation, missing values in employment_type column: 0


#### 6.3 Most Frequent Categorical Imputer

In [71]:
print(f'before imputation, missing values in gender column: {df["gender"].isnull().sum()}')
gender_imputer = SimpleImputer(strategy='most_frequent')
df['gender'] = gender_imputer.fit_transform(df[['gender']]).ravel()
print(f'after imputation, missing values in gender column: {df["gender"].isnull().sum()}')

before imputation, missing values in gender column: 47
after imputation, missing values in gender column: 0


#### 6.4 Missing Indicator + Random Sample Imputation

In [72]:
df_sample=df.copy()
print(f'before imputation, missing values in annual_income column: {df_sample["annual_income"].isnull().sum()}')
df_sample['annual_income_missing_ind'] = df_sample['annual_income'].isnull().astype(int)

null_count = df_sample['annual_income'].isnull().sum()
random_values = df_sample['annual_income'].dropna().sample(
    n=null_count, 
    random_state=42,
    replace=True).values

df_sample.loc[df_sample['annual_income'].isnull(), 'annual_income'] = random_values
print(f'after imputation, missing values in annual_income column: {df_sample["annual_income"].isnull().sum()}')
df_sample[['annual_income', 'annual_income_missing_ind']].head()

before imputation, missing values in annual_income column: 83
after imputation, missing values in annual_income column: 0


,annual_income,annual_income_missing_ind
0,841473.298565,0
1,179168.069828,0
2,454201.985323,1
3,311056.808239,0
4,378954.979964,1


#### 6.5 KNN Imputer

In [76]:
df_knn = df.copy()
numeric_columns =[ 'annual_income', 'loan_amount', 'credit_score']

print(f'before imputation, missing values in numeric columns:\n{df_knn[numeric_columns].isnull().sum()}')
knn_imputer = KNNImputer(n_neighbors=5)
df_knn[numeric_columns] = knn_imputer.fit_transform(df_knn[numeric_columns])
print(f'after imputation, missing values in numeric columns:\n{df_knn[numeric_columns].isnull().sum()}')

before imputation, missing values in numeric columns:
annual_income     83
loan_amount        0
credit_score     112
dtype: int64
after imputation, missing values in numeric columns:
annual_income    0
loan_amount      0
credit_score     0
dtype: int64


#### 6.6 MICE Algorithm

In [77]:
print(f'before imputation, missing values in numeric columns:\n{df[numeric_columns].isnull().sum()}')
iterative_imputer = IterativeImputer(max_iter=10, random_state=42)
df[numeric_columns] = iterative_imputer.fit_transform(df[numeric_columns])
print(f'after imputation, missing values in numeric columns:\n{df[numeric_columns].isnull().sum()}')

before imputation, missing values in numeric columns:
annual_income     83
loan_amount        0
credit_score     112
dtype: int64
after imputation, missing values in numeric columns:
annual_income    0
loan_amount      0
credit_score     0
dtype: int64


#### 6.7 Complete Case Analysis (dropping rows/columns)

## Part D: Outlier Handling

### 7. Detect and treat outliers using:

#### 7.1 Z-Score Method

#### 7.2 IQR Method

#### 7.3 Percentile Method

#### 7.4 Winsorization Method